## Description

This notebook includes the implementation of the second of the two main stages of the RAG pipeline: retrieval and generation. In addition to the implementation, we will test it using several standard types of questions

## Import libraries

In [1]:
from qdrant_client.models import Filter, FieldCondition, MatchValue, Prefetch, SparseVector, FusionQuery
from fastembed import SparseTextEmbedding, TextEmbedding
from qdrant_client import QdrantClient
import cohere
from dotenv import load_dotenv
import os
from google import genai
import math
from IPython.display import Markdown, display

In [2]:
load_dotenv()

True

## RAG pipeline implementation

For retrieval from the vector database, we will use a hybrid search approach that combines vector and keyword search. To generate embeddings, we will use the same models that were used during data indexing: BAAI/bge-small-en-v1.5 and Qdrant/minicoil-v1. In addition to the standard RAG pipeline, we will employ techniques such as multi-query retrieval and reranking.

For reranking, we will use the Cohere Rerank API. For multi-query generation and final answer generation, we will use the Gemini API. To improve the understanding of user queries, we will also extract any relevant years mentioned in the query and apply year-based filtering before performing the hybrid search.

The implementation is divided into the following functions:

In [3]:
def get_dense_embedding(dense_model, text):

    return next(dense_model.embed([text])).tolist()

In [4]:
def get_sparse_embedding(sparse_model, text):

    sparse_embedding = next(sparse_model.embed([text]))

    values = sparse_embedding.values.tolist()
    indices = sparse_embedding.indices.tolist()

    return values, indices

In [5]:
def retrieval(query, qdrant_client, dense_model, sparse_model, relevant_years, limit_vector_search=200, limit_keywords_search=200, 
                                                                                                                        limit_result=100):

    dense_embedding = get_dense_embedding(dense_model, query)
    sparse_embedding_values, sparse_embedding_indices = get_sparse_embedding(sparse_model, query)


    query_filter = None

    if relevant_years != []:

        query_filter = Filter(
            should=[
                FieldCondition(
                    key="year",
                    match=MatchValue(value=year)
                )
                for year in relevant_years
            ]
        )


    results = qdrant_client.query_points(
        collection_name="regulations",
        with_payload=True,
        prefetch=[
            Prefetch(
                query=dense_embedding,
                using="dense",
                limit=limit_vector_search,
                filter=query_filter
            ),
            Prefetch(
                query=SparseVector(
                    indices=sparse_embedding_indices,
                    values=sparse_embedding_values
                ),
                using="sparse",
                limit=limit_keywords_search,
                filter=query_filter
            ),
        ],
        query=FusionQuery(fusion="rrf"),
        limit=limit_result,
    )


    return results.points

In [6]:
def reranking(query, list_points, cohere_client, top_n):

    texts = []

    for point in list_points:

        text = f"""
        Regulation Type: {point.payload['regulation_type']} regulation
        Year: {point.payload['year']}
        Article: {point.payload['article']}
        Chapter: {point.payload['chapter']}

        {point.payload['content']}
        """.strip()

        texts.append(text)


    response = cohere_client.rerank(
        model="rerank-v4.0-pro",
        query=query,
        documents=texts,
        top_n=top_n,
    )
    
    reranking_result = [ list_points[item.index] for item in response.results ]


    return reranking_result

In [7]:
def generate_multi_queries(query, gemini_client, number_generated_queries=4):

    gemini_prompt = f"""
    Generate {number_generated_queries} alternative search queries for the following question.

    The queries should have the same meaning but use different wording.

    Question:
    {query}

    Return only the queries, one per line.
    """

    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=gemini_prompt
    )

    return [query] + response.text.split("\n")

In [8]:
def reciprocal_rank_fusion(result_lists, k=60):

    id_point = {}

    for points_list in result_lists:

        for idx, point in enumerate(points_list):

            id = point.id

            if id not in id_point:
                id_point[id] = point
                id_point[id].score = 0

            id_point[id].score += ( 1 / (k + (idx+1)) )


    rrf_result = sorted( list(id_point.items()), key=lambda x: x[1].score, reverse=True )
    rrf_result = list( dict(rrf_result).values() )


    return rrf_result     

In [9]:
def generate_answer(user_query, list_points, gemini_client):

    texts = []

    for point in list_points:

        text = f"""
        Regulation Type: {point.payload['regulation_type']} regulation
        Year: {point.payload['year']}
        Article: {point.payload['article']}
        Chapter: {point.payload['chapter']}

        {point.payload['content']}
        """.strip()

        texts.append(text)

    context = '\n\n'.join(texts)


    prompt = f"""
    You are an expert on FIA Formula 1 Regulations.

    Answer the question using ONLY the provided context.

    Rules:

    - Do not use outside knowledge.
    - If the answer is not available in the context, say:
    "The provided regulations do not contain enough information to answer this question."
    - For every factual statement, provide the corresponding year, regulation type, and article number.
    - If multiple articles are relevant, cite all of them.
    - Be precise, concise, and factual.
    - Prefer concise summaries over reproducing regulatory text.
    - Do not quote large portions of regulations unless necessary.
    - Use bullet points when comparing regulations.

    - When answering comparison questions, focus only on substantive regulatory changes.
    - Compare regulations based on the content and meaning of the rules, not article numbering.
    - Do not assume that two rules are equivalent because they share the same article number.
    - Do not assume that two rules are different because they have different article numbers.
    - Do not treat article renumbering, article references, cross-references, formatting changes, or wording changes as regulatory changes unless the actual meaning of the rule has changed.
    - Changes in article references, article numbering, and cross-references must not be reported as regulatory changes unless the provided context demonstrates a change in the underlying rule.
    - If the only observed difference is a change in article references, article numbering, or cross-references, classify the result as "Insufficient information for comparison" unless the content of the referenced regulations is available and demonstrates a substantive change.
    - A difference in wording or references alone is not sufficient evidence of a substantive regulatory change.
    - A regulatory change should be reported only when the rule's requirements, permissions, restrictions, procedures, obligations, or penalties have changed.
    - Before reporting a regulatory change, verify that the actual rule content differs between the years.

    - Structure comparison answers using the following sections:
    - Confirmed substantive regulatory changes
    - Confirmed unchanged regulations
    - Insufficient information for comparison

    - If no confirmed substantive regulatory changes are found, explicitly state:
    "No confirmed substantive regulatory changes were identified in the provided context."

    - Do not conclude that a rule was added, removed, or modified unless evidence from all relevant years is present in the context.
    - Missing articles in the retrieved context should be treated as insufficient information, not as evidence of a regulatory change.
    - If the available context is insufficient to compare a rule across years, classify it as "Insufficient information for comparison".

    - For comparison questions, summarize the outcome of the comparison before providing supporting details.
    - Focus on the most important findings rather than listing every matching article.

    Context:
    {context}

    Question:
    {user_query}
    """


    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )


    return response.text


In [10]:
def extract_years(query, gemini_client):

    prompt = f"""
    Extract Formula 1 regulation years mentioned in the question.

    If a range is requested, return all years in that range.

    Return ONLY a LIST of integers.

    Examples:

    Question:
    What were the Safety Car rules in 2021?

    Answer:
    [2021]

    Question:
    Compare the 2018 and 2026 regulations.

    Answer:
    [2018, 2026]

    Question:
    How did the regulations change from 2020 to 2026?

    Answer:
    [2020, 2021, 2022, 2023, 2024, 2025, 2026]

    Question:
    How did the regulations evolve between 2019 and 2022?

    Answer:
    [2019, 2020, 2021, 2022]

    Question:
    In which year was the fastest lap point removed?

    Answer:
    []

    Question:
    {query}
    """


    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt
    )


    return eval(response.text)

In [11]:
def rag_pipeline(user_query, gemini_client, qdrant_client, dense_model, sparse_model, cohere_client):

    relevant_years = extract_years(user_query, gemini_client)

    queries = generate_multi_queries(user_query, gemini_client)

    hybrid_search_results_lists = [ retrieval(query, qdrant_client, dense_model, sparse_model, relevant_years) for query in queries]
    
    rrf_result = reciprocal_rank_fusion(hybrid_search_results_lists)


    reranking_top_n = 25   # 20
    reranking_result = []

    if len(relevant_years) <= 1:
        reranking_result = reranking(user_query, rrf_result, cohere_client, reranking_top_n)
    else:
        top_n_per_year = math.ceil( reranking_top_n / len(relevant_years) ) 

        for year in relevant_years:

            year_specific_points = list( filter( lambda x: x.payload['year'] == year, rrf_result ) )
            reranking_year_res = reranking(user_query, year_specific_points, cohere_client, min(top_n_per_year, len(year_specific_points)))

            reranking_result += reranking_year_res


    final_answer = generate_answer(user_query, reranking_result, gemini_client)


    return final_answer, reranking_result

## Retrieval and generation testing

After implementing the RAG pipeline, it is necessary to evaluate its performance. For testing purposes, six standard categories of questions commonly asked about Formula 1 regulations were selected. These question types cover factual retrieval, regulatory comparison, historical analysis, reverse lookup, regulation traceability and scenario-based reasoning.

| Question Type | Example Question |
|---------------|------------------|
| Fact Retrieval | Under what conditions can the Safety Car be deployed during a race in 2024? |
| Comparison | How did the Safety Car regulations change between 2021 and 2022? |
| Historical Evolution | How have the Sprint regulations evolved from 2021 to 2026? |
| Reverse Lookup | In which year was the fastest lap point removed? |
| Regulation Traceability | Which articles in the 2026 Sporting Regulations govern Safety Car procedures? |
| Scenario-Based Reasoning | A lapped driver ignored multiple blue flags and did not allow the leading car to pass during a race in 2024. According to the Sporting Regulations, should the driver receive a penalty? Explain your reasoning and cite the relevant article(s). |

In [12]:
# function to display the retrieved relevant chunks

def print_retrieved_chunks(retrieved_chunks):

    for i, point in enumerate(retrieved_chunks):       
        p = point.payload

        print("=" * 120)
        print(f"Result #{i+1}")
        print(f"Year       : {p['year']}")
        print(f"Article    : {p['article']}")
        print(f"Chapter    : {p['chapter']}")
        print("-" * 120)
        print(p['content'])
        print()
    

In [13]:
# Initialize RAG pipeline components

dense_model = TextEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

sparse_model = SparseTextEmbedding(
    model_name="Qdrant/minicoil-v1"
)

qdrant_client = QdrantClient(path='../data/qdrant_storage')

cohere_client = cohere.ClientV2(api_key=os.getenv("COHERE_API_KEY"))

gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

### Question number 1: Fact Retrieval

In [17]:
user_query = "Under what conditions can the Safety Car be deployed during a race in 2024?"
answer, retrieved_chunks = rag_pipeline(user_query, gemini_client, qdrant_client, dense_model, sparse_model, cohere_client)

In [18]:
display(Markdown(answer))

The Safety Car may be deployed during a race in 2024 under the following conditions:

*   Upon the order of the clerk of the course, to neutralise a race. (2024, sporting regulation, Article 55.3)
*   Only if Competitors or officials are in immediate physical danger on or near the track. (2024, sporting regulation, Article 55.3)
*   When the circumstances are not such as to necessitate suspending the race. (2024, sporting regulation, Article 55.3)

In [19]:
print_retrieved_chunks(retrieved_chunks)

Result #1
Year       : 2024
Article    : 55.3
Chapter    : SAFETY CAR
------------------------------------------------------------------------------------------------------------------------
The safety car may be brought into operation to neutralise a sprint session or a race upon the order of the clerk of the course. It will be used only if Competitors or officials are in immediate physical danger on or near the track but the circumstances are not such as to necessitate suspending the sprint session or the race.

Result #2
Year       : 2024
Article    : 58.10
Chapter    : RESUMING A SPRINT SESSION OR A RACE
------------------------------------------------------------------------------------------------------------------------
The safety car will enter the pits after one (1) lap unless: a) The sprint session or the race is being resumed in wet conditions and the Race Director deems more than one lap is necessary, in which case the use of wet-weather tyres as specified under Article 30.

The system successfully identified the relevant regulation and returned a concise and accurate answer. The response correctly extracted the deployment conditions and provided precise references to the corresponding article and year. This demonstrates effective retrieval and citation capabilities for factual questions.

### Question number 2: Comparison

In [21]:
user_query = "How did the Safety Car regulations change between 2021 and 2022?"
answer, retrieved_chunks = rag_pipeline(user_query, gemini_client, qdrant_client, dense_model, sparse_model, cohere_client)

In [22]:
display(Markdown(answer))

The Safety Car regulations underwent several substantive changes between 2021 and 2022, primarily affecting the procedure for lapped cars overtaking, the application of safety car rules to different session types, and updates to referenced articles.

### Confirmed substantive regulatory changes

1.  **Safety Car Return After Lapped Cars Overtake (Regulation type: sporting regulation)**
    *   **2021 (Article 48.12):** The safety car was required to return to the pits at the end of the following lap *once the last lapped car had passed the leader*. The clerk of the course also had the explicit provision to send a "OVERTAKING WILL NOT BE PERMITTED" message if track conditions were unsuitable for overtaking.
    *   **2022 (Article 55.13):** The safety car is now required to return to the pits at the end of the following lap *once the message “LAPPED CARS MAY NOW OVERTAKE” has been sent to all Competitors using the official messaging system*. The provision allowing the clerk of the course to prohibit overtaking if track conditions were unsuitable was removed.

2.  **Safety Car Deployment on the Last Lap (Regulation type: sporting regulation)**
    *   **2021 (Article 48.15):** If the safety car was deployed on the last lap, cars would take the *end-of-race* signal.
    *   **2022 (Article 55.16):** If the safety car is deployed on the last lap, cars now take the *end-of-session* signal, broadening its application beyond just the race.

3.  **General Overtaking Exceptions During Safety Car (Regulation type: sporting regulation)**
    *   **2021 (Article 48.8b):** The list of articles permitting overtaking exceptions included Articles 41.1c), 48.12, 51.6, and 51.12.
    *   **2022 (Article 55.8b):** The list of articles permitting overtaking exceptions was updated to Articles 49.6, 55.13, 58.6, and 58.12. As the content of Article 48.12 / 55.13 has changed, this modifies the scope of permitted overtaking.

4.  **Leader's Position Behind Safety Car Exception (Regulation type: sporting regulation)**
    *   **2021 (Article 48.10):** Exceptions to the safety car remaining deployed until cars are lined up and the leader staying within ten car lengths were referenced as Article 48.12 and 48.13.
    *   **2022 (Article 55.10):** These exceptions were updated to reference Article 55.12 and 55.13. Since Article 48.12 / 55.13 has a substantive change, the conditions for these exceptions have also changed.

5.  **Sprint Session Terminology (Regulation type: sporting regulation)**
    *   **2021 (Articles 48.3, 48.13, 48.14):** Rules referred to "sprint qualifying session".
    *   **2022 (Articles 55.3, 55.14, 55.15):** The terminology was changed to "sprint session", indicating a change in the designation of the session to which these rules apply.

### Confirmed unchanged regulations

1.  **Safety Car Deployment Message and Signalling (Regulation type: sporting regulation)**
    *   **2021 (Article 48.4) and 2022 (Article 55.4):** The procedure for deploying the safety car, including the "SAFETY CAR DEPLOYED" message, "SC" display on FIA light panels, and waved yellow flags with "SC" boards at marshal posts, remained consistent.

### Insufficient information for comparison

1.  **Safety Car Reconnaissance Lap Exception (Regulation type: sporting regulation)**
    *   **2021 (Article 48.2) and 2022 (Article 55.2):** An exception to the safety car's initial procedure is referenced as "Article 42.1c)" in 2021 and "Article 49" in 2022. Without the content of these referenced articles, the nature of this change cannot be assessed.

2.  **Penalties for Minimum Time Violation under Safety Car (Regulation type: sporting regulation)**
    *   **2021 (Article 48.7) and 2022 (Article 55.7):** Penalties for failing to stay above the minimum time are referenced as "Article 47.3a), b), c) or d)" in 2021 and "Article 54.3a), 54.3b), 54.3c) or 54.3d)" in 2022. Without the content of these penalty articles, the nature of any change in penalties cannot be assessed.

3.  **Use of Pit Lane by Safety Car and Cars (Regulation type: sporting regulation)**
    *   **2021 (Article 48.11):** This article details conditions for cars and the safety car to use the pit lane.
    *   **2022 (Article 55.11):** This article is referenced in 2022 Article 55.8g) but its content is not provided, making a comparison impossible.

4.  **Safety Car Return after One Lap Conditions (Regulation type: sporting regulation)**
    *   **2021 (Article 51.10):** This article specifies conditions for the safety car entering the pits after one lap. No corresponding article from 2022 is provided in the context.

5.  **Virtual Safety Car Speed Reduction (Regulation type: sporting regulation)**
    *   **2021 (Article 49.5):** This article outlines requirements for speed reduction under a Virtual Safety Car. No corresponding VSC article from 2022 is provided in the context.

6.  **Safety Car Driver and Observer (Regulation type: sporting regulation)**
    *   **2022 (Article 55.1):** This article describes the roles of the FIA safety car driver and observer. No corresponding article from 2021 is provided in the context.

7.  **Formation Lap Overtaking Exceptions (Regulation type: sporting regulation)**
    *   **2022 (Article 49.6):** This article details specific circumstances where overtaking is permitted during a formation lap behind the safety car. No directly corresponding article from 2021 is provided in the context.

8.  **Formation Lap Procedure Behind Safety Car (Regulation type: sporting regulation)**
    *   **2022 (Article 49.2):** This article describes the procedure when the safety car leads the grid for a formation lap. No directly corresponding article from 2021 is provided in the context.

9.  **Safety Car Laps Counted Exception (Regulation type: sporting regulation)**
    *   **2021 (Article 48.14) and 2022 (Article 55.15):** An exception regarding how laps are counted when a specific procedure is followed is referenced as "Article 41.1c)" in 2021 and "Article 49" in 2022. Without the content of these referenced articles, the nature of this change cannot be assessed.

In [23]:
print_retrieved_chunks(retrieved_chunks)

Result #1
Year       : 2021
Article    : 48.12
Chapter    : SAFETY CAR
------------------------------------------------------------------------------------------------------------------------
If the clerk of the course considers it safe to do so, and the message "LAPPED CARS MAY NOW OVERTAKE" has been sent to all Competitors via the official messaging system, any cars that have been lapped by the leader will be required to pass the cars on the lead lap and the safety car. This will only apply to cars that were lapped at the time they crossed the Line at the end of the lap during which they crossed the first Safety Car line for the second time after the safety car was deployed. Having overtaken the cars on the lead lap and the safety car these cars should then proceed around the track at an appropriate speed, without overtaking, and make every effort to take up position at the back of the line of cars behind the safety car. Whilst they are overtaking, and in order to ensure this may be 

The system successfully identified both substantive regulatory changes and unchanged regulations. It correctly distinguished meaningful procedural modifications from simple article renumbering and grouped uncertain cases into an "Insufficient information" section. This demonstrates strong performance on regulatory comparison tasks.

### Question number 3: Historical Evolution

In [25]:
user_query = "How have the Sprint regulations evolved from 2021 to 2026?"
answer, retrieved_chunks = rag_pipeline(user_query, gemini_client, qdrant_client, dense_model, sparse_model, cohere_client)

In [26]:
display(Markdown(answer))

The Sprint regulations have undergone significant evolution from 2021 to 2026, encompassing changes in terminology, grid formation, penalty conditions, permitted work during session suspensions, and a major overhaul of the sprint weekend format with the introduction of Sprint Qualifying.

### Confirmed substantive regulatory changes

**From 2021 to 2022:**

*   **Terminology Shift:** The term "sprint qualifying session" used in 2021 (e.g., 2021 Sporting Regulation Article 6.5, Article 37.12, Article 46.1, Article 35.4, Article 47.3) was generally replaced with "sprint session" in 2022 (e.g., 2022 Sporting Regulation Article 54.3, Article 43.9, Article 41.2, Article 57.4, Article 40.3).
*   **Grid for the Race (2021 Sporting Regulation Article 35.4 vs. 2022 Sporting Regulation Article 41.3a)):**
    *   **2021:** The grid for the race was based on the final classification of the sprint qualifying session with a relatively straightforward application of grid penalties.
    *   **2022:** Introduced a substantially more detailed and complex procedure for drawing up the grid for the race based on the sprint session's final classification. This included specific steps for allocating grid positions, handling disqualified drivers, and applying cumulative grid penalties (2022 Sporting Regulation Article 41.3a)).
*   **Parc Fermé Regulations (2022 Sporting Regulation Article 40.3):**
    *   **2021:** The provided context does not contain specific parc fermé regulations for sprint qualifying sessions.
    *   **2022:** Introduced specific provisions for Competitions with a sprint session, allowing the replacement of parts that are different in design if they were previously used in a qualifying practice session or a race, with prior notification to the FIA.
*   **Sprint Session Starting Procedure (2022 Sporting Regulation Article 43.9):**
    *   **2021:** The provided context does not contain specific rules regarding practice starts or formation tightness during the formation lap of a sprint qualifying session.
    *   **2022:** Explicitly forbade practice starts and mandated that the formation must be kept as tight as possible during the formation lap of a sprint session.
*   **Suspension of Sprint Session/Race (2022 Sporting Regulation Article 57.4):**
    *   **2021:** The provided context does not detail permitted work on cars during a suspended sprint qualifying session or race.
    *   **2022:** Introduced a list of nine specific types of work permitted on cars once they have stopped in the fast lane during a suspended sprint session or race (e.g., starting the engine, adding compressed gases, changing wheels, repairing accident damage).
*   **Steward Penalties (2021 Sporting Regulation Article 47.3 vs. 2022 Sporting Regulation Article 54.3):**
    *   **2021:** Suspension from the driver's next Event was a possible penalty.
    *   **2022:** Changed the penalty to suspension from the driver's next Competition.

**From 2022 to 2023:**

*   **Sprint Session Distance and Duration (2023 Sporting Regulation Article 5.3):**
    *   **2022:** The provided context did not contain a dedicated article outlining the distance and maximum duration of a sprint session or the handling of suspensions.
    *   **2023:** Introduced a dedicated article defining the sprint session distance (least number of complete laps exceeding 100km) and duration rules. This included a one-hour time limit, an extension of up to 1.5 hours total with suspension time added, and a reduction in laps if the formation lap starts behind a safety car.
*   **Parc Fermé for Sprint Weekends (2023 Sporting Regulation Article 40.4):**
    *   **2022:** Allowed for replacement parts of different design at sprint events under certain conditions (2022 Sporting Regulation Article 40.3).
    *   **2023:** Expanded on the parc fermé regulations for sprint Competitions by specifically listing additional items that may be replaced with components of the same specification (e.g., brake friction material, engine exhaust system, engine oil filters, intake air filter, spark plugs).

**From 2023 to 2024:**

*   **Incident Penalties (2023 Sporting Regulation Article 54.3 vs. 2024 Sporting Regulation Article 54.3):**
    *   **2023:** A grid place penalty could be imposed if a driver was unable to serve a 5 or 10-second time penalty (a or b) due to "retirement from the sprint session or the race."
    *   **2024:** Expanded the condition for imposing a grid place penalty to include drivers unable to serve a 5 or 10-second time penalty (a or b) due to being "unclassified in the sprint session or the race," in addition to retirement.
*   **Tyre Allocation and Quantity (2024 Sporting Regulation Article 30.2):**
    *   **2023:** The provided context did not contain detailed tyre regulations specific to sprint session Competitions.
    *   **2024:** Introduced comprehensive regulations detailing the quantity, selection, and specification of tyres for Competitions where a sprint session is scheduled (e.g., 12 sets of dry-weather tyres, 5 intermediate, 2 wet-weather, with specific allocations for hard, medium, and soft compounds). This article also differentiated tyre allocations for Competitions with and without sprint sessions.
*   **Penalties for Overtaking during Resumption (2024 Sporting Regulation Article 58.9):**
    *   **2023:** The provided context did not contain specific penalties for unnecessary overtaking during the formation lap(s) when resuming a suspended session.
    *   **2024:** Introduced a new rule stating that stewards may impose a drive-through or ten-second stop-and-go penalty on any driver who unnecessarily overtakes another car during the lap (or laps) after suspension.

**From 2024 to 2025:**

*   **Sprint Session Maximum Duration Commencement (2025 Sporting Regulation Article 5.3):**
    *   **2024:** The context did not explicitly state when the maximum total sprint session time commences if the formation lap starts behind the safety car.
    *   **2025:** Explicitly clarified that if the formation lap for the sprint session is started behind the safety car, the maximum total sprint session time of one and a half (1.5) hours will commence when the green lights on the start gantry illuminate to signal the safety car's departure from the grid.
*   **Suspension Work (2024 Sporting Regulation Article 57.4 vs. 2025 Sporting Regulation Article 57.4):**
    *   **2024:** Listed nine specific types of work permitted on cars during suspension.
    *   **2025:** Added a tenth permitted activity: replenishing or replacing the cooling medium in the Driver Cooling System if a "Heat Hazard" has been declared.
*   **Grid Personnel Limit (2025 Sporting Regulation Article 43.6):**
    *   **2024:** The provided context did not contain specific limits on team personnel on the grid for sprint sessions.
    *   **2025:** Introduced a new rule limiting the number of team personnel for each Competitor to no more than sixteen (16) on the grid when the three (3) minute signal is shown for a sprint session.

**From 2025 to 2026:**

*   **Sprint Qualifying Format (2026 Sporting Regulation Article B2.2.2):**
    *   **2025:** The provided context describes sprint sessions but does not detail a specific "Sprint Qualifying" format.
    *   **2026:** Introduced a new, dedicated "Sprint Qualifying" format with three periods (SQ1, SQ2, SQ3) of specified durations (12, 10, and 8 minutes, respectively), including elimination rules for the slowest cars after SQ1 and SQ2.
*   **Sprint Qualifying Classification (2026 Sporting Regulation Article B2.2.3):**
    *   **2025:** The provided context does not detail a specific "Sprint Qualifying Classification" process.
    *   **2026:** Introduced comprehensive rules for determining the "Sprint Qualifying Classification," including ordering classified drivers based on lap times from SQ3, SQ2, and SQ1, rules for identical lap times, and criteria for "unclassified" drivers (e.g., 107% rule, no lap time, disqualification) and their relative ordering.
*   **Grid for the Sprint Session (2026 Sporting Regulation Article B2.3.4):**
    *   **2025:** The provided context (or 2022 Article 41.3a)) outlined simpler rules for forming the sprint grid based on qualifying results.
    *   **2026:** Introduced a significantly revised and highly detailed procedure for forming the grid for the Sprint. This new procedure incorporates the results of the new Sprint Qualifying, applies penalties based on cumulative unserved grid penalties from the previous twelve months using a multi-step allocation process for temporary and final grid positions for both penalised and unpenalised classified drivers, as well as unclassified drivers. It also specifies timelines for provisional and final grid publication and handling withdrawals.

### Confirmed unchanged regulations

*   **Practice Starts During Formation Lap (Sporting Regulation Article 43.9):** The prohibition of practice starts and the requirement for a tight formation during the formation lap for a sprint session remained unchanged in wording and meaning from 2022 through 2025 (2022 Sporting Regulation Article 43.9, 2023 Sporting Regulation Article 43.9, 2024 Sporting Regulation Article 43.9, 2025 Sporting Regulation Article 43.9).
*   **Most Steward Penalties (Sporting Regulation Article 54.3):** The general types of penalties that stewards may impose for incidents during a sprint session or race remained largely consistent in their definition and application from 2022 through 2025 (2022 Sporting Regulation Article 54.3, 2023 Sporting Regulation Article 54.3, 2024 Sporting Regulation Article 54.3, 2025 Sporting Regulation Article 54.3), with the exception of the additional condition for unclassified drivers introduced in 2024. The rule that if any of the seven specified penalties are imposed they shall not be subject to appeal also remained unchanged from 2021 (Sporting Regulation Article 47.3) through 2025 (Sporting Regulation Article 54.3).
*   **Sprint Session Duration (2025 Sporting Regulation Article 5.3 vs. 2026 Sporting Regulation Article B2.3.3):** The substantive rules for sprint session duration, including the maximum time, the addition of suspension length, and the commencement of the maximum time when a formation lap starts behind the safety car, remained consistent in meaning between 2025 and 2026, despite article renumbering.

### Insufficient information for comparison

*   **2021 Articles 6.5 (Points for suspended sessions) and 46.1 (Drivers leaving pit lane) have no direct equivalent or comparison points in the provided context for later years.**
*   **2021 Article 37.12 (Pit wall personnel for sprint qualifying start) has no direct equivalent in the provided context for later years for a direct comparison, other than the 2025 Article 43.6 which limits total personnel on the grid.**
*   **The 2021 Article 35.4 c), d), e), f) and 2022 Article 41.4 b), c), d), e), f) rules concerning specific details of race grid formation (e.g., for unclassified or penalized drivers) were not provided for 2023, 2024, 2025, or 2026, making a full comparison across all years impossible.**
*   **There is no context provided for 2026 regarding Incident Penalties (e.g., Article 54.3), Suspension of Sprint Session/Race (e.g., Article 57.4), Parc Fermé (e.g., Article 40.3/40.4), Tyre Regulations (e.g., Article 30.2), or Penalties for Overtaking During Resumption (e.g., Article 58.9). Therefore, these regulations cannot be compared from 2025 to 2026.**
*   **The specific content of numerous cross-referenced articles (e.g., Article 47.4b, Article 50, Article 42.1c, Article 23.2b, Article 28.3, Article 40.2g, Article 49, Article 5.4d, Article B5.16.1, Article B5.14.2) is not provided, meaning a full understanding of changes based solely on these cross-references is not possible.**

In [27]:
print_retrieved_chunks(retrieved_chunks)

Result #1
Year       : 2021
Article    : 47.3
Chapter    : INCIDENTS DURING THE SPRINT QUALIFYING SESSION OR THE RACE
------------------------------------------------------------------------------------------------------------------------
The stewards may impose any one of the penalties below on any driver involved in an Incident: a) A five second time penalty. The driver must enter the pit lane, stop in his pit stop position for at least five seconds and then re-join the sprint qualifying session or the race. The relevant driver may however elect not to stop, provided he carries out no further pit stop before the end of the sprint qualifying session or the race. In such cases five seconds will be added to the elapsed the sprint qualifying session or race time of the driver concerned. b) A ten second time penalty. The driver must enter the pit lane, stop in his pit stop position for at least ten seconds and then re-join the sprint qualifying session or the race. The relevant driver may

The system successfully identified the major stages in the evolution of the Sprint regulations from 2021 to 2026, correctly highlighting substantial regulatory changes such as the introduction of the new Sprint Qualifying format, revised grid formation procedures, and updates to Sprint weekend operations. While the response was comprehensive and factually accurate, it also included several secondary changes and sections with insufficient information that were not central to the user's question, making the answer more verbose than necessary. 

### Question number 4: Reverse Lookup

In [28]:
user_query = "In which year was the fastest lap point removed?"
answer, retrieved_chunks = rag_pipeline(user_query, gemini_client, qdrant_client, dense_model, sparse_model, cohere_client)

In [29]:
display(Markdown(answer))

The provided regulations do not contain enough information to answer this question. The context shows that a point for the fastest lap was awarded in all years from 2019 to 2024:

*   **2019, sporting regulation, Article 6.4**: "one point will be awarded to the driver who achieved the fastest lap of the race and to the constructor whose car he was driving, provided the lap time was achieved without incurring a penalty and he was in the top ten positions of the final race classification..."
*   **2020, sporting regulation, Article 6.4**: "one point will be awarded to the driver who achieved the fastest valid lap time of the race and to the constructor whose car he was driving, provided he was in the top ten positions of the final race classification..."
*   **2021, sporting regulation, Article 6.4**: "one point will be awarded to the driver who achieved the fastest valid lap time of the race and to the constructor whose car he was driving, provided he was in the top ten positions of the final race classification..."
*   **2022, sporting regulation, Article 6.4**: "one (1) point will be awarded to the driver who achieved the fastest valid lap time of the race and to the constructor whose car he was driving, provided he was in the top ten (10) positions of the final race classification..."
*   **2023, sporting regulation, Article 6.4**: "one (1) point will be awarded to the driver who achieved the fastest valid lap time of the race and to the constructor whose car he was driving, provided he was in the top ten (10) positions of the final race classification..."
*   **2024, sporting regulation, Article 6.4**: "one (1) point will be awarded to the driver who achieved the fastest valid lap time of the race and to the constructor whose car he was driving, provided he was in the top ten (10) positions of the final race classification..."

In [30]:
print_retrieved_chunks(retrieved_chunks)

Result #1
Year       : 2019
Article    : 6.4
Chapter    : WORLD CHAMPIONSHIP
------------------------------------------------------------------------------------------------------------------------
Points for both titles will be awarded at each Event according to the following scale : 1st : 25 points 2nd : 18 points 3rd : 15 points 4th : 12 points 5th : 10 points 6th : 8 points 7th : 6 points 8th : 4 points 9th : 2 points 10th : 1 point In addition to the above, one point will be awarded to the driver who achieved the fastest lap of the race and to the constructor whose car he was driving, provided the lap time was achieved without incurring a penalty and he was in the top ten positions of the final race classification (see Article 45). No point will be awarded if the fastest lap is achieved by a driver who was classified outside the top ten positions.

Result #2
Year       : 2021
Article    : 6.4
Chapter    : WORLD CHAMPIONSHIP
---------------------------------------------------------

The system was unable to answer the question because the required information was not available in the retrieved regulations. However, it correctly identified the limitation of the available context and avoided generating unsupported information. This demonstrates reliable handling of incomplete knowledge but also highlights a limitation of the current dataset coverage.

### Question number 5: Regulation Traceability

In [31]:
user_query = "Which articles in the 2026 Sporting Regulations govern Safety Car procedures?"
answer, retrieved_chunks = rag_pipeline(user_query, gemini_client, qdrant_client, dense_model, sparse_model, cohere_client)

In [32]:
display(Markdown(answer))

The 2026 Sporting Regulations govern Safety Car procedures in the following articles:

*   2026 Sporting Regulation Article B5.13.1 (Deployment of Safety Car)
*   2026 Sporting Regulation Article B5.13.2 (During a SC Deployment)
*   2026 Sporting Regulation Article B5.13.3 (Use of Pit Lane during a SC Deployment)
*   2026 Sporting Regulation Article B5.13.4 (Order of Cars Behind the SC)
*   2026 Sporting Regulation Article B5.13.5 (Duration of SC Period)
*   2026 Sporting Regulation Article B5.13.6 (Withdrawal of Safety Car)
*   2026 Sporting Regulation Article B5.13.7 (Each lap completed while the Safety Car is deployed)
*   2026 Sporting Regulation Article B5.13.8 (If the Safety Car is still deployed at the beginning of the last lap)
*   2026 Sporting Regulation Article B5.10.1 (Formation Lap(s) Behind Safety Car)
*   2026 Sporting Regulation Article B5.10.2 (Formation Lap(s) Behind Safety Car)
*   2026 Sporting Regulation Article B5.10.4 (Formation Lap(s) Behind Safety Car)
*   2026 Sporting Regulation Article B5.10.5 (Formation Lap(s) Behind Safety Car)
*   2026 Sporting Regulation Article B5.10.6 (Formation Lap(s) Behind Safety Car)
*   2026 Sporting Regulation Article B5.10.7 (Formation Lap(s) Behind Safety Car)
*   2026 Sporting Regulation Article B5.10.8 (Formation Lap(s) Behind Safety Car)
*   2026 Sporting Regulation Article B5.10.9 (Formation Lap(s) Behind Safety Car)
*   2026 Sporting Regulation Article B5.12.5 (The Safety Car will be used only if Competitors or officials are in immediate physical danger)
*   2026 Sporting Regulation Article B5.14.1 (All F1 Cars must enter the Pit Lane behind the Safety Car)
*   2026 Sporting Regulation Article B5.14.3 (All F1 Cars must enter the Pit Lane behind the Safety Car)
*   2026 Sporting Regulation Article B5.15.1 (The Safety Car will be positioned at the front of the line of F1 Cars)
*   2026 Sporting Regulation Article B5.2.1 (The Safety Car will leave the Pit Lane and take up position at the front of the grid)
*   2026 Sporting Regulation Article B5.15.3 (Overtaking during the laps behind the Safety Car at the resumption)

In [33]:
print_retrieved_chunks(retrieved_chunks)

Result #1
Year       : 2026
Article    : B5.13.4
Chapter    : TOTAL TIME CLASSIFIED SESSIONS (TTCS), Safety Car (SC)
------------------------------------------------------------------------------------------------------------------------
Order of Cars Behind the SC a. When instructed by the Race Director the green light on the Safety Car will be illuminated to signal to F1 Cars between it and the leader that they are required to pass. Once all such cars have passed the Safety Car the green light on the Safety Car will be extinguished to signal that overtaking is no longer permitted, with the exception of the cases listed in Article B5.13.2c. These F1 Cars will continue at reduced speed and without overtaking until they reach the queue of F1 Cars behind the Safety Car. b. If the Race Director considers track conditions are unsuitable for overtaking the message “OVERTAKING WILL NOT BE PERMITTED” will be sent to all Competitors. c. If the Race Director considers it safe to do so, the mess

The system successfully identified and organized the relevant articles governing Safety Car procedures in the 2026 Sporting Regulations. It covered both the core Safety Car provisions and related procedures, demonstrating effective retrieval of multiple interconnected regulations and accurate identification of the relevant articles.

### Question number 6: Scenario-Based Reasoning

In [14]:
user_query = """A lapped driver ignored multiple blue flags and did not allow the leading car to pass during a race in 2024.
According to the Sporting Regulations, should the driver receive a penalty? Explain your reasoning and cite the relevant article(s)."""

answer, retrieved_chunks = rag_pipeline(user_query, gemini_client, qdrant_client, dense_model, sparse_model, cohere_client)

In [15]:
display(Markdown(answer))

Yes, the driver could receive a penalty if the stewards determine their actions constitute a breach of regulations.

Reasoning:
*   A marshal uses a blue flag and/or a flashing blue light to warn drivers of approaching cars on the track (2024, Sporting Regulation, Article 53.1).
*   Drivers must not drive unnecessarily slowly, erratically, or in a manner which could be deemed potentially dangerous to other drivers (2024, Sporting Regulation, Article 33.4).
*   The Race Director may report any on-track incident or suspected breach of Sporting Regulations to the stewards, who then have the discretion to investigate (2024, Sporting Regulation, Article 54.1).
*   The stewards have the discretion to decide if any driver involved in an incident should be penalised, particularly if the driver was wholly or predominantly to blame (2024, Sporting Regulation, Article 54.2a).
*   If the stewards decide to impose a penalty, they may choose from the range of penalties listed in Article 54.3 (2024, Sporting Regulation, Article 54.3).

Therefore, if the stewards deem that ignoring blue flags and not allowing a leading car to pass constitutes driving "unnecessarily slowly, erratically or in a manner which could be deemed potentially dangerous" or an "Incident" where the driver was predominantly to blame, a penalty may be imposed.

In [16]:
print_retrieved_chunks(retrieved_chunks)

Result #1
Year       : 2024
Article    : 58.9
Chapter    : RESUMING A SPRINT SESSION OR A RACE
------------------------------------------------------------------------------------------------------------------------
Either of the penalties prescribed in Article 54.3c) or 54.3d) will be imposed on any driver who, in the opinion of the stewards, unnecessarily overtook another car during the lap (or laps).

Result #2
Year       : 2024
Article    : 33.2
Chapter    : DRIVING
------------------------------------------------------------------------------------------------------------------------
Drivers must observe the provisions of the Code relating to driving behaviour on circuits at all times.

Result #3
Year       : 2024
Article    : 26.1
Chapter    : GENERAL SAFETY
------------------------------------------------------------------------------------------------------------------------
Official instructions will be given to drivers by means of the signals laid out in the Code. Competitors

The system successfully applied the relevant regulations to a realistic racing scenario instead of simply retrieving information. It correctly identified the applicable articles, explained that the final decision rests with the stewards, and supported its reasoning with appropriate regulatory references. This demonstrates the system's potential to assist in regulatory interpretation and decision support.

## Overall testing conclusion

The evaluation shows that the proposed RAG pipeline performs well across different types of regulatory questions. The system accurately retrieves relevant regulations, provides article-level references, compares regulations across seasons, identifies historical changes, and applies regulations to realistic racing scenarios.

The main strengths of the system are accurate retrieval, transparent citation of sources, and reliable handling of comparison and scenario-based questions. When the required information is not available, the system correctly reports insufficient information instead of generating unsupported answers.

The main limitation is that responses to broad historical questions may include secondary regulatory changes or unnecessary details, making the answers more verbose than required. Additionally, the quality of the answers depends on the completeness of the indexed regulations.

Overall, the results demonstrate that the proposed Formula 1 Regulatory Intelligence Assistant can effectively support regulatory search, comparison, interpretation, and decision support based on FIA Sporting Regulations.